#### Import Libraries

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("All good 🚀")

All good 🚀


##### Load the dataset

In [132]:
model_df = pd.read_csv("../data/processed/bed_occupancy_modelling_dataset.csv")

#### Create more features

In [168]:
model_df["over_capacity"] = (
    model_df["occupancy_rate"] > 1
).astype(int)

In [169]:
model_df["available_beds"] = (
    model_df["available_beds"]
    .clip(lower=0)
)

#### Sort Date Column

In [170]:
model_df["date"] = pd.to_datetime(
    model_df["date"]
)

In [171]:
model_df = model_df.sort_values(
    [
        "hospital_id",
        "ward",
        "date"
    ]
)

In [172]:
model_df.duplicated(
    [
        "hospital_id",
        "ward",
        "date"
    ]
).sum()

0

##### Check Date Coverage - continuous dates for forecasting

In [173]:
model_df.groupby(
    [
        "hospital_id",
        "ward"
    ]
)["date"].agg(
    ["min","max","count"]
)

min        max  count
hospital_id ward                                                
HHN-BIR-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-EDI-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-02  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-MAN-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731

##### Add Time-Series Features

##### Lag 1 - Yesterday's occupancy

In [174]:
model_df["lag_1_occupied_beds"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .shift(1)
)

##### Lag 7 - One week ago (Previous 7 days)

In [175]:
model_df["lag_7_occupied_beds"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .shift(7)
)

##### 7-day moving average

In [176]:
model_df["rolling_7_day_avg"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .transform(
        lambda x:
        x.rolling(7).mean()
    )
)

##### Rolling 7-day Average

In [177]:
model_df["rolling_7_day_avg"] = (
    model_df.groupby(["hospital_id", "ward"])["occupied_beds"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

##### Rolling 7-day Standard deviation

In [178]:
model_df["rolling_7_day_std"] = (
    model_df.groupby(["hospital_id", "ward"])["occupied_beds"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

##### Day of week

In [179]:
model_df["day_of_week"] = (
    model_df["date"]
    .dt.dayofweek
)

##### Month

In [180]:
model_df["month"] = (
    model_df["date"].dt.month
)

##### Length of Stay by day

In [181]:
model_df["avg_los_days"] = (
    model_df["avg_los_hours"] / 24
).round(1)

##### Day-to-Day Change

In [182]:
model_df["occupied_beds_change"] = (
    model_df.groupby(
        [
            "hospital_id",
            "ward"
        ]
    )["occupied_beds"]
    .diff()
)

##### Weekend

In [183]:
model_df["is_weekend"] = (
    model_df["date"].dt.dayofweek >= 5
).astype(int)

##### Fill Lag NA Values

In [184]:
lag_columns = [
    "lag_1_occupied_beds",
    "lag_7_occupied_beds",
    "rolling_7_day_avg",
    "occupied_beds_change"
]

model_df[lag_columns] = (
    model_df[lag_columns]
    .fillna(0)
)

In [185]:
print("Shape:", model_df.shape)

model_df.info()

Shape: (29240, 48)
<class 'pandas.DataFrame'>
RangeIndex: 29240 entries, 0 to 29239
Data columns (total 48 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   level_0               29240 non-null  int64         
 1   index                 29240 non-null  int64         
 2   hospital_id           29240 non-null  str           
 3   ward                  29240 non-null  str           
 4   bed_type              29240 non-null  str           
 5   date                  29240 non-null  datetime64[us]
 6   total_beds            29240 non-null  int64         
 7   staffed_beds          29240 non-null  int64         
 8   occupied_beds         29240 non-null  int64         
 9   closed_beds           29240 non-null  int64         
 10  occupancy_rate        29240 non-null  float64       
 11  available_beds        29240 non-null  int64         
 12  daily_admissions      29240 non-null  float64       
 13  avg_los_

In [186]:
missing = (
    model_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

rolling_7_day_std       280
level_0                   0
day                       0
occupancy_rate_model      0
bed_pressure              0
lag_1_occupancy           0
lag_7_occupancy           0
rolling_7_day_avg         0
day_of_week               0
occupancy_change          0
year                      0
month                     0
week_of_year              0
index                     0
quarter                   0
is_weekend                0
available_bed_pct         0
staff_shortage            0
emergency_ratio           0
surgery_per_staff         0
lag_1_occupied_beds       0
lag_7_occupied_beds       0
avg_los_days              0
over_capacity             0
total_bed_capacity        0
specialty_emphasis        0
available_beds            0
hospital_id               0
ward                      0
bed_type                  0
date                      0
total_beds                0
staffed_beds              0
occupied_beds             0
closed_beds               0
occupancy_rate      

In [187]:
duplicates = model_df.duplicated(
    subset=[
        "hospital_id",
        "ward",
        "bed_type",
        "date"
    ]
).sum()

print("Duplicate records:", duplicates)

Duplicate records: 0


In [188]:
print(
    model_df["occupied_beds"].describe()
)

count    29240.000000
mean        14.114603
std          8.732209
min          0.000000
25%          9.000000
50%         13.000000
75%         19.000000
max         43.000000
Name: occupied_beds, dtype: float64


#### Feature Engineering

##### Calendar Features

In [189]:
model_df["date"] = pd.to_datetime(model_df["date"])

model_df["year"] = model_df["date"].dt.year

model_df["month"] = model_df["date"].dt.month

model_df["day"] = model_df["date"].dt.day

model_df["day_of_week"] = model_df["date"].dt.dayofweek

model_df["week_of_year"] = model_df["date"].dt.isocalendar().week.astype(int)

model_df["quarter"] = model_df["date"].dt.quarter

model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)

##### Admission Pressure

In [190]:
model_df["admission_pressure"] = (
    model_df["daily_admissions"] /
    model_df["staffed_beds"]
).round(2)

##### Bed Pressure

In [191]:
model_df["bed_pressure"] = (
    model_df["occupied_beds"]
    -
    model_df["staffed_beds"]
)

##### Available Bed Percentage

In [192]:
model_df["available_bed_pct"] = (
    model_df["available_beds"]
    /
    model_df["staffed_beds"]
).round(1)

##### Staff Shortage

In [193]:
model_df["staff_shortage"] = (
    model_df["planned_staff"]
    -
    model_df["actual_staff"]
)

##### Emergency Admission Ratio

In [194]:
model_df["emergency_ratio"] = (
    model_df["emergency_admissions"]
    /
    model_df["daily_admissions"]
).round(1)

In [195]:
model_df["emergency_ratio"] = (
    model_df["emergency_ratio"]
    .fillna(0)
)

##### Surgery Pressure

In [196]:
model_df["surgery_per_staff"] = (
    model_df["scheduled_surgeries"]
    /
    model_df["actual_staff"]
).round(1)

model_df["surgery_per_staff"] = (
    model_df["surgery_per_staff"]
    .fillna(0)
)

In [197]:
model_df.head()

,level_0,index,hospital_id,ward,bed_type,date,total_beds,staffed_beds,occupied_beds,closed_beds,occupancy_rate,available_beds,daily_admissions,avg_los_hours,emergency_admissions,planned_staff,actual_staff,staffing_ratio,daily_ed_arrivals,scheduled_surgeries,hospital_name,city,tier,specialty_emphasis,total_bed_capacity,over_capacity,occupancy_rate_model,bed_pressure,lag_1_occupancy,lag_7_occupancy,rolling_7_day_avg,day_of_week,occupancy_change,year,month,day,week_of_year,quarter,is_weekend,available_bed_pct,staff_shortage,emergency_ratio,surgery_per_staff,lag_1_occupied_beds,lag_7_occupied_beds,rolling_7_day_std,avg_los_days,occupied_beds_change,admission_pressure
0,0,0,HHN-BIR-01,Cardiology Ward,Standard,2024-01-01,42,42,3,0,0.1,39,6.0,84.0,6.0,20,20,1.0,80,0.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.1,-39,0.0,0.0,0.0,0,0.0,2024,1,1,1,1,0,0.9,0,1.0,0.0,0.0,0.0,NaN,3.5,0.0,0.14
1,1,1,HHN-BIR-01,Cardiology Ward,Standard,2024-01-02,42,41,10,0,0.2,31,10.0,102.0,9.0,20,17,0.8,83,3.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.2,-31,0.1,0.0,0.0,1,0.1,2024,1,2,1,1,0,0.8,3,0.9,0.2,3.0,0.0,NaN,4.2,7.0,0.24
2,2,2,HHN-BIR-01,Cardiology Ward,Standard,2024-01-03,42,42,18,0,0.4,24,7.0,70.0,6.0,20,20,1.0,72,2.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.4,-24,0.2,0.0,0.0,2,0.2,2024,1,3,1,1,0,0.6,0,0.9,0.1,10.0,0.0,NaN,2.9,8.0,0.17
3,3,3,HHN-BIR-01,Cardiology Ward,Standard,2024-01-04,42,41,19,0,0.5,22,7.0,66.0,5.0,20,20,1.0,95,9.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.5,-22,0.4,0.0,0.0,3,0.1,2024,1,4,1,1,0,0.5,0,0.7,0.4,18.0,0.0,NaN,2.8,1.0,0.17
4,4,4,HHN-BIR-01,Cardiology Ward,Standard,2024-01-05,42,42,22,0,0.5,20,6.0,94.0,2.0,20,18,0.9,80,14.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.5,-20,0.5,0.0,0.0,4,0.0,2024,1,5,1,1,0,0.5,2,0.3,0.8,19.0,0.0,NaN,3.9,3.0,0.14


##### Save time series dataset

In [201]:
model_df.to_csv(
    "../data/processed/bed_occupancy_timeseries.csv",
    index=False
)